# Datetime Code Walkthrough: From Basics to Time-Series Pitfalls

Hands-on companion notebook for your live demo. Run from top to bottom.

This notebook is designed for analytics students who already heard the presentation and now need practical instincts for real datetime work in Python and pandas.

Learning goals:
- Create and inspect Python date/time objects
- Understand and test timezone-naive vs timezone-aware behavior
- Convert between ISO 8601, Unix timestamps, and pandas datetimes
- Use timedeltas safely
- Engineer calendar features (`year`, `month`, `weekday`, `hour`) from timestamp columns
- Explore U.S. electricity demand patterns where timezone and DST details matter

## 1. Setup

In [ ]:
# If needed in a fresh environment:
# %pip install numpy pandas matplotlib
from datetime import date, time, datetime, timedelta, timezone
from zoneinfo import ZoneInfo
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("pandas", pd.__version__)

## 2. Python date/time types

In [ ]:
today = date.today()
clock_time = time(8, 15)
meeting = datetime(2026, 11, 2, 8, 15)
duration = timedelta(minutes=90)
print(today, clock_time, meeting, meeting + duration)
print("ISO:", meeting.isoformat())

Try it: create your own date and time, combine them, and add 90 minutes.

In [ ]:
sample = datetime(2026, 11, 2, 8, 15, 30, tzinfo=timezone.utc)
print("type:", type(sample))
print("year/month/day:", sample.year, sample.month, sample.day)
print("hour/minute/second:", sample.hour, sample.minute, sample.second)
print("tzinfo:", sample.tzinfo)

# Quick discovery: methods that often matter in analytics work
core_methods = [m for m in dir(sample) if m in {"isoformat", "timestamp", "astimezone", "strftime", "replace"}]
print("core methods:", core_methods)

Try it: call `dir(sample)` and inspect at least five methods you have not used before. Which ones seem most useful for analytics pipelines?

## 3. Naive vs timezone-aware datetimes

In [ ]:
naive = datetime(2026, 11, 2, 9, 0)
chicago = datetime(2026, 11, 2, 9, 0, tzinfo=ZoneInfo("America/Chicago"))

print("naive:", naive, naive.tzinfo)
print("Chicago aware:", chicago, chicago.utcoffset())
print("UTC equivalent:", chicago.astimezone(timezone.utc))
print("Paris equivalent:", chicago.astimezone(ZoneInfo("Europe/Paris")))

# Intentional demo: Python blocks direct ordering comparisons between naive and aware datetimes
try:
    print("naive < chicago?", naive < chicago)
except TypeError as e:
    print("TypeError:", e)

# Correct approach: make both aware in a known zone (or both in UTC) before comparing
naive_as_chicago = naive.replace(tzinfo=ZoneInfo("America/Chicago"))
print("Comparable now:", naive_as_chicago < chicago)

`replace(tzinfo=...)` attaches an interpretation to a wall-clock reading; it does not convert clock time. `astimezone(...)` converts an already-aware instant while preserving the instant. For comparisons, put both values on the same semantic footing first (typically UTC or the same IANA timezone).

## 4. Daylight saving: missing and repeated times

In [ ]:
zone = ZoneInfo("America/Chicago")
first = datetime(2026, 11, 1, 1, 30, tzinfo=zone, fold=0)
second = datetime(2026, 11, 1, 1, 30, tzinfo=zone, fold=1)
for x in (first, second):
    print(x.isoformat(), "UTC ->", x.astimezone(timezone.utc))

The fall-back transition repeats an hour; `fold` distinguishes its two occurrences. The spring-forward transition skips local clock readings. Local wall-clock text alone may not identify a unique instant.

## 5. Elapsed duration vs calendar time

In [ ]:
start = datetime(2026, 3, 7, 12, tzinfo=zone)
plus_24h = (start.astimezone(timezone.utc) + timedelta(hours=24)).astimezone(zone)
next_local_noon = datetime(2026, 3, 8, 12, tzinfo=zone)
print("start:", start)
print("24 elapsed hours later:", plus_24h)
print("same local time next day:", next_local_noon)
print("elapsed hours to next local noon:", (next_local_noon.astimezone(timezone.utc)-start.astimezone(timezone.utc)).total_seconds()/3600)